# AB_Rossmann ETL Implementation Guide

## Data Warehouse / Business Intelligence Case Study

This notebook provides a step-by-step guide through the ETL (Extract, Transform, Load) process for the Rossmann Store Sales Data Warehouse project.

**Objective**: Transform raw sales data into a dimensional data model following strict transformation and validation rules.

## 1. Business Context

### Problem Statement
Rossmann (German drugstore chain) needs to optimize organizational performance through a Data Warehouse/BI solution that provides:

- **Key Performance Indicators (KPIs)**:
  - Total revenue (EUR)
  - Customer volume
  - Average spend per customer

- **Business Questions**:
  - Which store types generate highest sales?
  - How do promotions affect sales?
  - How do holidays influence turnover?
  - Does competitor distance impact store performance?

## 2. Setup and Configuration

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
import logging
import os
from sqlalchemy import create_engine, text
from datetime import datetime

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

print('Libraries imported successfully!')

In [ ]:
# Load environment variables from .env file
load_dotenv()

# Database connection settings (Supabase connection pooler)
SD_HOST = os.getenv('SD_HOST')
SD_PORT = os.getenv('SD_PORT')
SD_DBNAME = os.getenv('SD_DBNAME')
SD_USER = os.getenv('SD_USER')
SD_PASSWORD = os.getenv('SD_PASSWORD')
SD_SSLMODE = os.getenv('SD_SSLMODE')

# Create database connection string for PostgreSQL pooled connection
db_connection_string = f'postgresql://{SD_USER}:{SD_PASSWORD}@{SD_HOST}:{SD_PORT}/{SD_DBNAME}?sslmode={SD_SSLMODE}'
engine = create_engine(db_connection_string, echo=False, pool_size=5)

# Test database connection
try:
    with engine.connect() as connection:
        connection.execute(text('SELECT 1'))
        print('Database connection successful!')
        print(f'Connected to: {SD_HOST}:{SD_PORT}/{SD_DBNAME}')
except Exception as e:
    print(f'Database connection failed: {str(e)}')

# Set data path for CSV files
data_path = Path('data')
print(f'Data path: {data_path}')

## 3. Extract Phase - Load Raw Data

In [ ]:
def load_raw_data():
    """Load raw data from CSV files"""
    logging.info('Loading raw data from CSV files...')
    store_path = data_path / 'store.csv'
    train_path = data_path / 'train.csv'
    
    df_store = pd.read_csv(store_path)
    df_train = pd.read_csv(train_path, low_memory=False)
    
    logging.info(f'Store records loaded: {len(df_store)}')
    logging.info(f'Training records loaded: {len(df_train)}')
    
    return df_store, df_train

df_store, df_train = load_raw_data()

print('\n=== STORE DATA ===')
print(f'Shape: {df_store.shape}')
print(f'Columns: {df_store.columns.tolist()}')
print('\nFirst 3 rows:')
print(df_store.head(3))

In [ ]:
print('\n=== TRAIN DATA ===')
print(f'Shape: {df_train.shape}')
print(f'Columns: {df_train.columns.tolist()}')
print('\nFirst 3 rows:')
print(df_train.head(3))

## 4. Data Validation Phase

### 4.1 Empty Values Check

In [ ]:
def validate_empty_values(df_store, df_train):
    """Check for empty/null values"""
    print('\n=== EMPTY VALUES CHECK ===')
    
    print('\nStore data missing values:')
    store_nulls = df_store.isnull().sum()
    print(store_nulls[store_nulls > 0] if (store_nulls > 0).any() else 'No missing values')
    
    print('\nTrain data missing values:')
    train_nulls = df_train.isnull().sum()
    print(train_nulls[train_nulls > 0] if (train_nulls > 0).any() else 'No missing values')
    
    # Critical validation: Date cannot be null
    if df_train['Date'].isnull().any():
        rows_with_null_dates = df_train[df_train['Date'].isnull()].shape[0]
        logging.warning(f'{rows_with_null_dates} rows have null dates and will be excluded')
        df_train = df_train.dropna(subset=['Date'])
    
    # Validation: SchoolHoliday and StateHoliday should not be null
    if df_train['SchoolHoliday'].isnull().any():
        logging.warning('Null values found in SchoolHoliday, filling with 0')
        df_train['SchoolHoliday'] = df_train['SchoolHoliday'].fillna(0)
    
    if df_train['StateHoliday'].isnull().any():
        logging.warning('Null values found in StateHoliday, filling with 0')
        df_train['StateHoliday'] = df_train['StateHoliday'].fillna('0')
    
    return df_store, df_train

df_store, df_train = validate_empty_values(df_store, df_train)

### 4.2 Invalid Data Check

In [ ]:
def validate_invalid_data(df_store, df_train):
    """Check for invalid data values"""
    print('\n=== INVALID DATA CHECK ===')
    
    initial_count = len(df_train)
    
    # Check for negative sales
    neg_sales = (df_train['Sales'] < 0).sum()
    if neg_sales > 0:
        logging.warning(f'{neg_sales} rows have negative sales')
        df_train = df_train[df_train['Sales'] >= 0]
    
    # Check for negative customers
    neg_customers = (df_train['Customers'] < 0).sum()
    if neg_customers > 0:
        logging.warning(f'{neg_customers} rows have negative customers')
        df_train = df_train[df_train['Customers'] >= 0]
    
    # Check for negative competition distance
    neg_distance = (df_store['CompetitionDistance'] < 0).sum()
    if neg_distance > 0:
        logging.warning(f'{neg_distance} stores have negative competition distance')
        df_store = df_store[df_store['CompetitionDistance'] >= 0]
    
    # Check for future dates
    df_train['Date'] = pd.to_datetime(df_train['Date'])
    future_dates = (df_train['Date'] > datetime.now()).sum()
    if future_dates > 0:
        logging.warning(f'{future_dates} rows have future dates')
        df_train = df_train[df_train['Date'] <= datetime.now()]
    
    # Check promotion year is not greater than date year
    df_store['Promo2SinceYear'] = pd.to_numeric(df_store['Promo2SinceYear'], errors='coerce')
    invalid_promo_years = 0
    # This will be validated after join
    
    rows_removed = initial_count - len(df_train)
    if rows_removed > 0:
        logging.info(f'Removed {rows_removed} rows due to invalid data')
    
    return df_store, df_train

df_store, df_train = validate_invalid_data(df_store, df_train)

### 4.3 Missing Keys Check

In [ ]:
def validate_missing_keys(df_store, df_train):
    """Check if Store ID is present in both sources"""
    print('\n=== MISSING KEYS CHECK ===')
    
    store_ids_in_store = set(df_store['Store'].unique())
    store_ids_in_train = set(df_train['Store'].unique())
    
    missing_in_store = store_ids_in_train - store_ids_in_store
    missing_in_train = store_ids_in_store - store_ids_in_train
    
    if missing_in_store:
        logging.warning(f'Store IDs in train but not in store: {len(missing_in_store)} stores')
        df_train = df_train[~df_train['Store'].isin(missing_in_store)]
    
    if missing_in_train:
        logging.warning(f'Store IDs in store but not in train: {len(missing_in_train)} stores')
    
    print(f'Total stores in store dataset: {len(store_ids_in_store)}')
    print(f'Stores with training data: {len(store_ids_in_store - missing_in_train)}')
    
    return df_store, df_train

df_store, df_train = validate_missing_keys(df_store, df_train)

### 4.4 Duplicate Check

In [ ]:
def validate_duplicates(df_train):
    """Check for duplicate entries"""
    print('\n=== DUPLICATE CHECK ===')
    
    # Check for duplicate Date + Store combinations
    duplicates_date_store = df_train.duplicated(subset=['Date', 'Store']).sum()
    if duplicates_date_store > 0:
        logging.warning(f'{duplicates_date_store} duplicate Date + Store combinations found')
        df_train = df_train.drop_duplicates(subset=['Date', 'Store'], keep='first')
    else:
        print('No duplicate Date + Store combinations found')
    
    print(f'Training records after duplicate removal: {len(df_train)}')
    
    return df_train

df_train = validate_duplicates(df_train)

## 5. Data Preparation Phase

### 5.1 Join Data Sources

In [ ]:
def prepare_data(df_store, df_train):
    """Join store and train datasets"""
    print('\n=== DATA PREPARATION - JOIN ===')
    
    # Join datasets on Store ID
    df_merged = df_train.merge(df_store, left_on='Store', right_on='Store', how='inner')
    
    print(f'Records after join: {len(df_merged)}')
    print(f'Columns in merged data: {len(df_merged.columns)}')
    
    return df_merged

df_merged = prepare_data(df_store, df_train)
print('\nMerged data sample:')
print(df_merged.head(2))

### 5.2 Standardize Data Types and Rename Columns

In [ ]:
def standardize_and_rename(df):
    """Standardize data types and rename columns"""
    print('\n=== STANDARDIZE DATA TYPES AND RENAME ===')
    
    df = df.copy()
    
    # Standardize Date format to YYYY-MM-DD
    df['Date'] = pd.to_datetime(df['Date']).dt.strftime('%Y-%m-%d')
    
    # Convert CompetitionDistance to float
    df['CompetitionDistance'] = pd.to_numeric(df['CompetitionDistance'], errors='coerce').astype('float64')
    
    # Convert Customers to integer
    df['Customers'] = pd.to_numeric(df['Customers'], errors='coerce').astype('int64')
    
    # Convert Sales to float
    df['Sales'] = pd.to_numeric(df['Sales'], errors='coerce').astype('float64')
    
    # Rename columns
    rename_dict = {
        'Store': 'store_id',
        'StoreType': 'store_type',
        'Assortment': 'assortment',
        'Promo2': 'promotion',
        'Promo2SinceYear': 'promotion_year',
        'PromoInterval': 'promotion_interval',
        'CompetitionDistance': 'distance',
        'CompetitionOpenSinceYear': 'comp_open_year',
        'CompetitionOpenSinceMonth': 'comp_open_month',
        'Date': 'full_date',
        'StateHoliday': 'state_holiday',
        'SchoolHoliday': 'school_holiday',
        'Sales': 'turnover',
        'Customers': 'nr_customers'
    }
    
    df = df.rename(columns=rename_dict)
    
    print('Column renaming completed')
    print(f'Columns after rename: {[c for c in df.columns if c in rename_dict.values()]}')
    
    return df

df_merged = standardize_and_rename(df_merged)

### 5.3 Map Holiday Values

In [ ]:
def map_holiday_values(df):
    """Map state holiday values to integers"""
    print('\n=== MAP HOLIDAY VALUES ===')
    
    df = df.copy()
    
    # Map StateHoliday: 0=None, a=Public(1), b=Easter(2), c=Christmas(3)
    state_holiday_map = {'0': 0, 'a': 1, 'b': 2, 'c': 3}
    df['state_holiday'] = df['state_holiday'].astype(str).map(state_holiday_map).fillna(0).astype('int64')
    
    # Ensure SchoolHoliday is binary (0 or 1)
    df['school_holiday'] = df['school_holiday'].astype('int64')
    
    print('Holiday values mapped')
    print(f'State holiday unique values: {sorted(df["state_holiday"].unique())}')
    print(f'School holiday unique values: {sorted(df["school_holiday"].unique())}')
    
    return df

df_merged = map_holiday_values(df_merged)

## 6. Transform Phase - Create Dimension Tables

### 6.1 Create dim_store

In [ ]:
def create_dim_store(df):
    """Create store dimension"""
    print('\n=== CREATE DIM_STORE ===')
    
    dim_store = df[['store_id', 'store_type', 'assortment']].drop_duplicates().reset_index(drop=True)
    
    # Map assortment codes to full names
    assort_map = {'a': 'Basic', 'b': 'Extra', 'c': 'Extended'}
    dim_store['assortment'] = dim_store['assortment'].map(assort_map)
    
    # Validate assortment
    if dim_store['assortment'].isnull().any():
        logging.warning(f'{dim_store["assortment"].isnull().sum()} rows have invalid assortment')
        dim_store = dim_store.dropna(subset=['assortment'])
    
    dim_store = dim_store.sort_values('store_id').reset_index(drop=True)
    
    print(f'dim_store records: {len(dim_store)}')
    print(f'Store types: {dim_store["store_type"].unique()}')
    print(f'Assortments: {dim_store["assortment"].unique()}')
    
    return dim_store

dim_store = create_dim_store(df_merged)
print('\nFirst 5 records:')
print(dim_store.head())

### 6.2 Create dim_competition

In [ ]:
def create_dim_competition(df):
    """Create competition dimension"""
    print('\n=== CREATE DIM_COMPETITION ===')
    
    comp_data = []
    
    for store_id in df['store_id'].unique():
        store_data = df[df['store_id'] == store_id].iloc[0]
        
        # Determine if competitor is open
        is_open = 1 if pd.notna(store_data['comp_open_year']) else 0
        
        comp_data.append({
            'distance': float(store_data['distance']) if pd.notna(store_data['distance']) else 0.0,
            'open': is_open,
            'open_year': int(store_data['comp_open_year']) if pd.notna(store_data['comp_open_year']) else 0,
            'open_month': int(store_data['comp_open_month']) if pd.notna(store_data['comp_open_month']) else 0
        })
    
    dim_competition = pd.DataFrame(comp_data).reset_index(drop=True)
    
    print(f'dim_competition records: {len(dim_competition)}')
    print(f'\nCompetition distance statistics:')
    print(dim_competition['distance'].describe())
    
    return dim_competition

dim_competition = create_dim_competition(df_merged)
print('\nFirst 5 records:')
print(dim_competition.head())

### 6.3 Create dim_promotion with PromoInterval splitting

In [ ]:
def create_dim_promotion(df):
    """Create promotion dimension with PromoInterval splitting"""
    print('\n=== CREATE DIM_PROMOTION (with interval splitting) ===')
    
    promo_data = []
    
    for store_id in df['store_id'].unique():
        store_data = df[df['store_id'] == store_id].iloc[0]
        
        # Get promotion status
        has_promotion = int(store_data['promotion']) if pd.notna(store_data['promotion']) else 0
        promo_year = int(store_data['promotion_year']) if pd.notna(store_data['promotion_year']) else 0
        promo_interval = str(store_data['promotion_interval']) if pd.notna(store_data['promotion_interval']) else ''
        
        # If no promotion, interval should be empty
        if has_promotion == 0:
            promo_interval = ''
        
        # Split PromoInterval by month
        if promo_interval and promo_interval != 'nan':
            months = [m.strip() for m in str(promo_interval).split(',')]
            interval_str = ','.join(months)
        else:
            interval_str = ''
        
        promo_data.append({
            'promotion': has_promotion,
            'promotion_year': promo_year,
            'promotion_interval': interval_str
        })
    
    dim_promotion = pd.DataFrame(promo_data).reset_index(drop=True)
    
    print(f'dim_promotion records: {len(dim_promotion)}')
    print(f'Stores with active promotions: {(dim_promotion["promotion"] == 1).sum()}')
    print(f'\nUnique promotion intervals:')
    unique_intervals = dim_promotion[dim_promotion['promotion_interval'] != '']['promotion_interval'].unique()
    for interval in unique_intervals[:5]:
        print(f'  {interval}')
    
    return dim_promotion

dim_promotion = create_dim_promotion(df_merged)
print('\nFirst 10 records:')
print(dim_promotion.head(10))

### 6.4 Create dim_time with date derivations

In [ ]:
def create_dim_time(df):
    """Create time dimension with date derivations"""
    print('\n=== CREATE DIM_TIME ===')
    
    # Get unique dates
    unique_dates = pd.to_datetime(df['full_date']).unique()
    unique_dates = sorted(unique_dates)
    
    time_data = []
    
    for date in unique_dates:
        day_data = df[pd.to_datetime(df['full_date']) == date]
        
        # Get holiday info (should be consistent for all stores on same day)
        school_holiday = int(day_data['school_holiday'].iloc[0])
        state_holiday = int(day_data['state_holiday'].iloc[0])
        
        # Validate that school_holiday and state_holiday are 0 or 1
        school_holiday = 1 if school_holiday > 0 else 0
        state_holiday = state_holiday if state_holiday in [0, 1, 2, 3] else 0
        
        time_data.append({
            'full_date': date.strftime('%Y-%m-%d'),
            'year': date.year,
            'month': date.month,
            'day': date.day,
            'school_holiday': school_holiday,
            'state_holiday': state_holiday
        })
    
    dim_time = pd.DataFrame(time_data).reset_index(drop=True)
    
    print(f'dim_time records: {len(dim_time)}')
    print(f'Date range: {dim_time["full_date"].min()} to {dim_time["full_date"].max()}')
    print(f'School holidays: {(dim_time["school_holiday"] == 1).sum()}')
    print(f'State holidays: {(dim_time["state_holiday"] > 0).sum()}')
    
    return dim_time

dim_time = create_dim_time(df_merged)
print('\nFirst 10 records:')
print(dim_time.head(10))

## 7. Create Fact Table with Business Rules

In [ ]:
def create_fact_sales(df, dim_store, dim_competition, dim_promotion, dim_time):
    """Create fact table with business rules applied"""
    print('\n=== CREATE FACT_SALES TABLE ===')
    
    df = df.copy()
    
    # Create mappings from dimension tables
    store_id_map = {idx + 1: row['store_id'] for idx, (_, row) in enumerate(dim_store.iterrows())}
    
    # Create fact table
    fact = pd.DataFrame()
    fact['store_id'] = df['store_id']
    
    # Add dimension IDs (in real scenario, these would be sequential IDs from database)
    fact['competition_id'] = df['store_id'].apply(lambda x: df[df['store_id'] == x].index[0] + 1)
    fact['promotion_id'] = df['store_id'].apply(lambda x: df[df['store_id'] == x].index[0] + 1)
    fact['time_id'] = df.reset_index().apply(lambda x: x['index'], axis=1)
    
    # Add measures
    fact['turnover'] = df['turnover']
    fact['nr_customers'] = df['nr_customers']
    
    # Apply business rule: Calculate turnover_per_customer
    fact['turnover_per_customer'] = 0.0
    mask = fact['nr_customers'] > 0
    fact.loc[mask, 'turnover_per_customer'] = (
        (fact.loc[mask, 'turnover'] / fact.loc[mask, 'nr_customers']).round(2)
    )
    
    # Apply business rules
    print('\nApplying business rules...')
    
    # Rule 1: Turnover must be >= 0
    neg_turnover = (fact['turnover'] < 0).sum()
    if neg_turnover > 0:
        logging.warning(f'{neg_turnover} rows have negative turnover')
        fact = fact[fact['turnover'] >= 0]
    
    # Rule 2: nr_customers must be >= 0
    neg_customers = (fact['nr_customers'] < 0).sum()
    if neg_customers > 0:
        logging.warning(f'{neg_customers} rows have negative customers')
        fact = fact[fact['nr_customers'] >= 0]
    
    # Rule 3: If nr_customers = 0, turnover_per_customer should be 0
    fact.loc[fact['nr_customers'] == 0, 'turnover_per_customer'] = 0.0
    
    # Rule 4: Each fact row must reference valid dimension keys
    # Validate store_id exists in dim_store
    valid_store_ids = set(dim_store['store_id'])
    invalid_stores = (~fact['store_id'].isin(valid_store_ids)).sum()
    if invalid_stores > 0:
        logging.warning(f'{invalid_stores} rows have invalid store_id')
        fact = fact[fact['store_id'].isin(valid_store_ids)]
    
    print(f'\nfact_sales records: {len(fact)}')
    print(f'Date range: (from merged data)')
    print(f'Total turnover: EUR {fact["turnover"].sum():,.2f}')
    print(f'Total customers: {fact["nr_customers"].sum():,}')
    print(f'Average turnover per transaction: EUR {fact["turnover"].mean():,.2f}')
    print(f'Average turnover per customer: EUR {fact[fact["nr_customers"] > 0]["turnover_per_customer"].mean():,.2f}')
    
    return fact

fact_sales = create_fact_sales(df_merged, dim_store, dim_competition, dim_promotion, dim_time)
print('\nFirst 10 records:')
print(fact_sales.head(10))

## 8. Data Quality Summary

In [ ]:
print('\n' + '='*70)
print('DATA QUALITY AND TRANSFORMATION SUMMARY')
print('='*70)

print(f'\nDimension Tables:')
print(f'  dim_store:        {len(dim_store):6} records')
print(f'  dim_competition:  {len(dim_competition):6} records')
print(f'  dim_promotion:    {len(dim_promotion):6} records')
print(f'  dim_time:         {len(dim_time):6} records')

print(f'\nFact Table:')
print(f'  fact_sales:       {len(fact_sales):6} records')

print(f'\nData Quality Checks:')
print(f'  No null store_id:           {(fact_sales["store_id"].notnull().all())}')
print(f'  No negative turnover:       {(fact_sales["turnover"] >= 0).all()}')
print(f'  No negative customers:      {(fact_sales["nr_customers"] >= 0).all()}')
print(f'  Valid turnover_per_customer:{(fact_sales["turnover_per_customer"] >= 0).all()}')

print(f'\nDimension Table Checks:')
print(f'  No null values in dim_store:      {dim_store.isnull().sum().sum() == 0}')
print(f'  No null values in dim_time:       {dim_time.isnull().sum().sum() == 0}')
print(f'  No null values in dim_competition:{dim_competition.isnull().sum().sum() == 0}')
print(f'  No null values in dim_promotion:  {dim_promotion.isnull().sum().sum() == 0}')

print('\n' + '='*70)

## 9. Load Phase - Upload to Database

In [ ]:
def upload_to_database(df, table_name, if_exists='append'):
    """Upload dataframe to PostgreSQL database using SQLAlchemy"""
    try:
        logging.info(f'Uploading {len(df)} rows to {table_name}...')
        df.to_sql(table_name, engine, if_exists=if_exists, index=False, method='multi')
        logging.info(f'Successfully uploaded {len(df)} rows to {table_name}')
        return True
    except Exception as e:
        logging.error(f'Error uploading to {table_name}: {str(e)}')
        raise e

print('Upload function defined.')

In [ ]:
def clear_table(table_name):
    """Clear all records from a table"""
    try:
        with engine.connect() as connection:
            connection.execute(text(f'DELETE FROM {table_name}'))
            connection.commit()
            logging.info(f'Cleared table: {table_name}')
    except Exception as e:
        logging.warning(f'Could not clear table {table_name}: {str(e)}')

print('Clear function defined.')

In [ ]:
# Uncomment to upload dimensions
# print('\nUploading dimension tables...')
# upload_to_database(dim_store, 'dim_store', if_exists='replace')
# upload_to_database(dim_competition, 'dim_competition', if_exists='replace')
# upload_to_database(dim_promotion, 'dim_promotion', if_exists='replace')
# upload_to_database(dim_time, 'dim_time', if_exists='replace')
# print('Dimension tables uploaded')

# Uncomment to upload fact table
# print('\nUploading fact table...')
# upload_to_database(fact_sales, 'fact_sales', if_exists='replace')
# print('Fact table uploaded')

print('Uncomment upload commands to execute database load.')

## 10. Complete ETL Workflow

In [ ]:
def run_complete_etl():
    """Run complete ETL process with all validations and transformations"""
    try:
        print('\n' + '='*70)
        print('COMPLETE ETL PIPELINE - EXTRACT, VALIDATE, TRANSFORM, LOAD')
        print('='*70)
        
        # EXTRACT
        print('\n>>> EXTRACT PHASE')
        df_store, df_train = load_raw_data()
        
        # VALIDATE
        print('\n>>> VALIDATION PHASE')
        df_store, df_train = validate_empty_values(df_store, df_train)
        df_store, df_train = validate_invalid_data(df_store, df_train)
        df_store, df_train = validate_missing_keys(df_store, df_train)
        df_train = validate_duplicates(df_train)
        
        # PREPARE
        print('\n>>> DATA PREPARATION PHASE')
        df_merged = prepare_data(df_store, df_train)
        df_merged = standardize_and_rename(df_merged)
        df_merged = map_holiday_values(df_merged)
        
        # TRANSFORM
        print('\n>>> TRANSFORM PHASE')
        dim_store = create_dim_store(df_merged)
        dim_competition = create_dim_competition(df_merged)
        dim_promotion = create_dim_promotion(df_merged)
        dim_time = create_dim_time(df_merged)
        fact_sales = create_fact_sales(df_merged, dim_store, dim_competition, dim_promotion, dim_time)
        
        # LOAD
        print('\n>>> LOAD PHASE')
        upload_to_database(dim_store, 'dim_store', if_exists='replace')
        upload_to_database(dim_competition, 'dim_competition', if_exists='replace')
        upload_to_database(dim_promotion, 'dim_promotion', if_exists='replace')
        upload_to_database(dim_time, 'dim_time', if_exists='replace')
        upload_to_database(fact_sales, 'fact_sales', if_exists='replace')
        
        print('\n' + '='*70)
        print('ETL PIPELINE COMPLETED SUCCESSFULLY')
        print('='*70)
        
    except Exception as e:
        logging.error(f'ETL process failed: {str(e)}')
        raise e

# Uncomment to run the complete ETL pipeline
# run_complete_etl()

print('Complete ETL pipeline function defined.')
print('Uncomment run_complete_etl() to execute the full pipeline.')

## 11. Verification and Analysis

In [ ]:
print('\n' + '='*70)
print('BUSINESS ANALYSIS AND METRICS')
print('='*70)

print(f'\nSales Summary:')
print(f'  Total turnover:             EUR {fact_sales["turnover"].sum():,.2f}')
print(f'  Total customers:            {fact_sales["nr_customers"].sum():,}')
print(f'  Average turnover per day:   EUR {fact_sales["turnover"].mean():,.2f}')
print(f'  Average customers per day:  {fact_sales["nr_customers"].mean():.0f}')
print(f'  Average per customer:       EUR {fact_sales[fact_sales["nr_customers"] > 0]["turnover_per_customer"].mean():,.2f}')

print(f'\nStore Analysis:')
print(f'  Total stores:               {len(dim_store)}')
print(f'  Store types:                {dim_store["store_type"].nunique()}')
print(f'  Assortments:                {dim_store["assortment"].nunique()}')

print(f'\nCompetition Analysis:')
print(f'  Avg distance to competitor: {dim_competition["distance"].mean():.0f}m')
print(f'  Stores with competitors:    {(dim_competition["open"] == 1).sum()}')

print(f'\nPromotion Analysis:')
print(f'  Stores with promotions:     {(dim_promotion["promotion"] == 1).sum()}')

print(f'\nTime Period Analysis:')
print(f'  Date range:                 {dim_time["full_date"].min()} to {dim_time["full_date"].max()}')
print(f'  Total days:                 {len(dim_time)}')
print(f'  School holidays:            {(dim_time["school_holiday"] == 1).sum()} days')
print(f'  State holidays:             {(dim_time["state_holiday"] > 0).sum()} days')